In [ ]:
import json

# Đọc cấu trúc file notebook để kiểm tra các yêu cầu
notebook_path = '/content/Bai 4. MLP với dữ liệu ảnh.ipynb'
with open(notebook_path, 'r', encoding='utf-8') as f:
    nb_data = json.load(f)

# Hiển thị tóm tắt các cell trong notebook
for i, cell in enumerate(nb_data.get('cells', [])):
    cell_type = cell.get('cell_type', '')
    source = ''.join(cell.get('source', []))
    print(f"Cell {i} ({cell_type}):")
    print(source[:200] + ('...' if len(source) > 200 else ''))
    print('-' * 50)

Cell 0 (markdown):
# BÀI 4. XÂY DỰNG MÔ HÌNH MLP CHO DỮ LIỆU ẢNH
## Trọng tâm: Workflow Deep Learning hoàn chỉnh

**Mục tiêu bài thực hành**

Sau bài này, sinh viên có thể:

1. Hiểu dữ liệu ảnh được biểu diễn dưới dạng ...
--------------------------------------------------
Cell 1 (markdown):
## 1. Chuẩn bị môi trường và thư viện

Trong Deep Learning, chúng ta thường cần:

- thư viện xử lý số liệu;
- thư viện trực quan hóa;
- framework Deep Learning;
- công cụ đánh giá mô hình.
--------------------------------------------------
Cell 2 (code):
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import confusion_matrix, classification_report, ConfusionMa...
--------------------------------------------------
Cell 3 (markdown):
### Hướng dẫn đọc kết quả

- `TensorFlow version` cho biết phiên bản framework đang sử dụng.
- `GPU available` cho biết Colab có nhận GPU hay không.
- Với Fashion-MNIST và MLP nh

In [ ]:
import json
import copy

# Đọc file notebook gốc
notebook_path = '/content/Bai 4. MLP với dữ liệu ảnh.ipynb'
with open(notebook_path, 'r', encoding='utf-8') as f:
    nb_data = json.load(f)

# Định nghĩa các cell bài tập mới để nối vào cuối notebook
exercise_cells = [
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "# LỜI GIẢI CHI TIẾT CÁC BÀI TẬP MỞ RỘNG\n",
            "Dưới đây là phần code chạy thực nghiệm và so sánh cho cả 4 bài tập."
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Bài 1: Thay đổi kiến trúc (1, 2, 3 Hidden Layers)\n",
            "Chúng ta sẽ xây dựng hàm tạo mô hình linh hoạt và tiến hành so sánh số lượng tham số, thời gian huấn luyện và độ chính xác."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "import time\n",
            "import matplotlib.pyplot as plt\n",
            "import tensorflow as tf\n",
            "from tensorflow.keras import layers, models\n",
            "\n",
            "# Hàm tạo mô hình MLP tùy biến số lượng hidden layers\n",
            "def build_mlp(num_hidden_layers=1, units=128):\n",
            "    model = models.Sequential()\n",
            "    model.add(layers.Flatten(input_shape=(28, 28)))\n",
            "    for _ in range(num_hidden_layers):\n",
            "        model.add(layers.Dense(units, activation='relu'))\n",
            "    model.add(layers.Dense(10, activation='softmax'))\n",
            "    return model\n",
            "\n",
            "# Chuẩn bị lại dữ liệu nhanh để chạy thực nghiệm\n",
            "(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()\n",
            "X_train, X_test = X_train / 255.0, X_test / 255.0\n",
            "\n",
            "results_arch = {}\n",
            "\n",
            "for layers_count in [1, 2, 3]:\n",
            "    print(f'\n--- Training MLP with {layers_count} Hidden Layer(s) ---')\n",
            "    model = build_mlp(num_hidden_layers=layers_count)\n",
            "    model.compile(optimizer='adam',\n",
            "                  loss='sparse_categorical_crossentropy',\n",
            "                  metrics=['accuracy'])\n",
            "    \n",
            "    # Đếm số lượng tham số\n",
            "    params = model.count_params()\n",
            "    \n",
            "    start_time = time.time()\n",
            "    history = model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.1, verbose=1)\n",
            "    train_time = time.time() - start_time\n",
            "    \n",
            "    val_acc = history.history['val_accuracy'][-1]\n",
            "    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)\n",
            "    \n",
            "    results_arch[layers_count] = {\n",
            "        'params': params,\n",
            "        'train_time': train_time,\n",
            "        'val_acc': val_acc,\n",
            "        'test_acc': test_acc\n",
            "    }\n",
            "\n",
            "# In bảng so sánh kết quả\n",
            "import pandas as pd\n",
            "df_arch = pd.DataFrame(results_arch).T\n",
            "df_arch.index.name = 'Num Hidden Layers'\n",
            "display(df_arch)"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Bài 2: Thử nghiệm các mức Learning Rate\n",
            "Chúng ta thử nghiệm các mức learning rate: `1e-2` (0.01), `1e-3` (0.001), và `1e-4` (0.001) để xem ảnh hưởng lên quá trình hội tụ."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "learning_rates = [1e-2, 1e-3, 1e-4]\n",
            "histories_lr = {}\n",
            "\n",
            "for lr in learning_rates:\n",
            "    print(f'\n--- Training with Learning Rate: {lr} ---')\n",
            "    model = build_mlp(num_hidden_layers=2)\n",
            "    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)\n",
            "    model.compile(optimizer=optimizer,\n",
            "                  loss='sparse_categorical_crossentropy',\n",
            "                  metrics=['accuracy'])\n",
            "    \n",
            "    history = model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.1, verbose=1)\n",
            "    histories_lr[lr] = history.history\n",
            "\n",
            "# Trực quan hóa đường cong học tập\n",
            "plt.figure(figsize=(12, 5))\n",
            "for lr in learning_rates:\n",
            "    plt.plot(histories_lr[lr]['val_loss'], label=f'LR {lr}')\n",
            "plt.title('Độ hao hụt trên tập Validation (Validation Loss) theo Learning Rate')\n",
            "plt.xlabel('Epoch')\n",
            "plt.ylabel('Loss')\n",
            "plt.legend()\n",
            "plt.show()"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Bài 3: Thử nghiệm kích thước Batch Size\n",
            "So sánh kích thước batch size: 32, 128, và 512."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "batch_sizes = [32, 128, 512]\n",
            "results_batch = {}\n",
            "\n",
            "for bs in batch_sizes:\n",
            "    print(f'\n--- Training with Batch Size: {bs} ---')\n",
            "    model = build_mlp(num_hidden_layers=2)\n",
            "    model.compile(optimizer='adam',\n",
            "                  loss='sparse_categorical_crossentropy',\n",
            "                  metrics=['accuracy'])\n",
            "    \n",
            "    start_time = time.time()\n",
            "    history = model.fit(X_train, y_train, epochs=5, batch_size=bs, validation_split=0.1, verbose=1)\n",
            "    duration = time.time() - start_time\n",
            "    \n",
            "    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)\n",
            "    \n",
            "    results_batch[bs] = {\n",
            "        'Train Time (s)': duration,\n",
            "        'Test Accuracy': test_acc\n",
            "    }\n",
            "\n",
            "df_batch = pd.DataFrame(results_batch).T\n",
            "df_batch.index.name = 'Batch Size'\n",
            "display(df_batch)"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Bài 4: Thêm kỹ thuật Dropout chống Overfitting\n",
            "Thêm lớp Dropout với tỷ lệ loại bỏ là 0.2 giữa các tầng ẩn để tăng tính tổng quát hóa của mô hình."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Xây dựng mô hình với lớp Dropout\n",
            "model_dropout = models.Sequential([\n",
            "    layers.Flatten(input_shape=(28, 28)),\n",
            "    layers.Dense(256, activation='relu'),\n",
            "    layers.Dropout(0.2),  # Tỷ lệ dropout 20%\n",
            "    layers.Dense(128, activation='relu'),\n",
            "    layers.Dropout(0.2),\n",
            "    layers.Dense(10, activation='softmax')\n",
            "])\n",
            "\n",
            "model_dropout.compile(optimizer='adam',\n",
            "                      loss='sparse_categorical_crossentropy',\n",
            "                      metrics=['accuracy'])\n",
            "\n",
            "history_dropout = model_dropout.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=1)\n",
            "\n",
            "# Vẽ biểu đồ loss so sánh train và validation để kiểm tra Overfitting\n",
            "plt.figure(figsize=(10, 5))\n",
            "plt.plot(history_dropout.history['loss'], label='Train Loss')\n",
            "plt.plot(history_dropout.history['val_loss'], label='Val Loss')\n",
            "plt.title('Độ hao hụt khi áp dụng Dropout')\n",
            "plt.xlabel('Epoch')\n",
            "plt.ylabel('Loss')\n",
            "plt.legend()\n",
            "plt.show()"
        ]
    }
]

# Nối thêm các cell lời giải vào cấu trúc notebook gốc
nb_data['cells'].extend(exercise_cells)

# Ghi đè lại nội dung mới vào file notebook
with open(notebook_path, 'w', encoding='utf-8') as f:
    json.dump(nb_data, f, ensure_ascii=False, indent=2)

print("Đã hoàn thành thêm toàn bộ lời giải chi tiết cho 4 bài tập vào file notebook của bạn!")

Đã hoàn thành thêm toàn bộ lời giải chi tiết cho 4 bài tập vào file notebook của bạn!
